In [1]:
# ============================================================
# FINAL MASTER DATASET (USING RAW FILES)
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Load RAW datasets (these contain ALL variables)
# ------------------------------------------------------------
LFS_PATH = "../1_data/raw/labour_force_raw.csv.csv"
JV_PATH  = "../1_data/raw/job vacancies.csv"

lfs = pd.read_csv(LFS_PATH)
jv  = pd.read_csv(JV_PATH)

# ------------------------------------------------------------
# 2. Filter LFS to total population (same as StatsCan clean)
# ------------------------------------------------------------
lfs = lfs[
    (lfs["Gender"] == "Total - Gender") &
    (lfs["Age group"] == "15 years and over") &
    (lfs["Data type"] == "Seasonally adjusted")
].copy()

# ------------------------------------------------------------
# 3. Pivot LFS (THIS TIME VALUE EXISTS)
# ------------------------------------------------------------
lfs_wide = (
    lfs
        .pivot_table(
            index=["REF_DATE"],
            columns="Labour force characteristics",
            values="VALUE",
            aggfunc="sum"
        )
        .reset_index()
)

# Rename REF_DATE → month to match JV
lfs_wide = lfs_wide.rename(columns={"REF_DATE": "month"})

# ------------------------------------------------------------
# 4. Prepare Job Vacancies
# ------------------------------------------------------------
jv = jv.rename(columns={"VALUE": "vacancies"})
jv = jv[["REF_DATE", "vacancies"]].rename(columns={"REF_DATE": "month"})

# ------------------------------------------------------------
# 5. Merge (PURE MERGE, NO CLEANING)
# ------------------------------------------------------------
final_master_df = lfs_wide.merge(
    jv,
    on="month",
    how="left"
)

# ------------------------------------------------------------
# 6. Save FINAL dataset
# ------------------------------------------------------------
final_master_df.to_csv(
    "../1_data/processed/final_master_dataset.csv",
    index=False
)

final_master_df.to_excel(
    "../1_data/processed/final_master_dataset.xlsx",
    index=False,
    sheet_name="final_master"
)


In [3]:
# ============================================================
# SPRINT 2 – STEP 1: STRUCTURAL DATA AUDIT
# User Story: US3 – Data Cleaning & Alignment
# Task: 3.1 Inspect Missing Values
# ============================================================

import pandas as pd

# Load raw master dataset (DO NOT MODIFY)
df = pd.read_excel("../1_data/processed/final_master_dataset.xlsx")

# Basic structure
print("Shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns)

# Data types
print("\nData types:")
print(df.dtypes)

# Missing values
print("\nMissing values by column:")
print(df.isna().sum())

# Duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Preview
df.head()


Shape (rows, columns): (117, 6)

Column names:
Index(['month', 'Employment', 'Labour force', 'Population', 'Unemployment',
       'vacancies'],
      dtype='object')

Data types:
month            object
Employment      float64
Labour force    float64
Population      float64
Unemployment    float64
vacancies       float64
dtype: object

Missing values by column:
month           0
Employment      0
Labour force    0
Population      0
Unemployment    0
vacancies       6
dtype: int64

Duplicate rows: 0


,month,Employment,Labour force,Population,Unemployment,vacancies
0,2015-04,17819.4,19156.6,29000.4,1337.2,440425.0
1,2015-05,17843.7,19168.4,29020.2,1324.7,407040.0
2,2015-06,17836.7,19163.3,29048.8,1326.6,398910.0
3,2015-07,17863.3,19199.2,29074.3,1336.0,376570.0
4,2015-08,17889.0,19247.3,29099.5,1358.3,369190.0


In [5]:
# ============================================================
# SPRINT 2 – TASK 3.2: MONTHLY TIMELINE ALIGNMENT CHECK
# ============================================================



# Convert month to datetime for validation ONLY
df['month'] = pd.to_datetime(df['month'])

# Sort chronologically
df = df.sort_values('month').reset_index(drop=True)

# Check start and end dates
print("Start month:", df['month'].min())
print("End month:", df['month'].max())

# Check monthly frequency
expected_months = pd.date_range(
    start=df['month'].min(),
    end=df['month'].max(),
    freq='MS'
)

actual_months = df['month']

missing_months = expected_months.difference(actual_months)

print("\nMissing months:")
print(missing_months)

# Confirm one row per month
print("\nUnique months:", df['month'].nunique())
print("Total rows:", df.shape[0])


Start month: 2015-04-01 00:00:00
End month: 2024-12-01 00:00:00

Missing months:
DatetimeIndex([], dtype='datetime64[ns]', freq='MS')

Unique months: 117
Total rows: 117


In [6]:
# Convert month to pandas Period (monthly)
df['month'] = pd.to_datetime(df['month']).dt.to_period('M')

# Check result
df.head()


,month,Employment,Labour force,Population,Unemployment,vacancies
0,2015-04,17819.4,19156.6,29000.4,1337.2,440425.0
1,2015-05,17843.7,19168.4,29020.2,1324.7,407040.0
2,2015-06,17836.7,19163.3,29048.8,1326.6,398910.0
3,2015-07,17863.3,19199.2,29074.3,1336.0,376570.0
4,2015-08,17889.0,19247.3,29099.5,1358.3,369190.0


### Handling of Missing Job Vacancy Data

The Job Vacancy and Wage Survey (JVWS) contains missing observations for the period
April 2020 to September 2020. This gap corresponds to the initial COVID-19 shock,
during which vacancy data collection and publication were disrupted.

To preserve the integrity of the time series and avoid introducing undocumented
assumptions, these observations were retained as missing values (NA). No imputation
or smoothing was applied at this stage. The handling of these missing values is
explicitly documented and will be addressed in later analytical phases if required.



In [7]:
# ============================================================
# SPRINT 2 – TASK 3.4: SAVE CLEANED DATASET (v2)
# ============================================================



# Convert month to string for CSV saving (YYYY-MM)
df['month'] = df['month'].astype(str)

# Save cleaned dataset (v2)
df.to_csv(
    "../1_data/processed/dataset_clean.csv",
    index=False
)



### Labour Market Imbalance Indicator (θ)

To measure labour market imbalance, this project defines an indicator (θ) as the
ratio of labour demand to labour supply:

theta = job vacancies / unemployment

Interpretation:
- θ > 1 indicates labour shortages (high demand relative to supply)
- θ < 1 indicates labour surplus (low demand relative to supply)

This indicator is computed at a monthly national level and is used throughout
descriptive, predictive, and scenario-based analyses.


In [8]:
# ============================================================
# US4 – TASK 4.2: COMPUTE LABOUR MARKET IMBALANCE INDICATOR (θ)
# ============================================================

import pandas as pd
import numpy as np

# Load cleaned dataset (v2)
df = pd.read_csv("../1_data/processed/dataset_clean.csv")

# Ensure month stays as Period (do not change format)
df['month'] = df['month'].astype(str)

# Compute theta safely
df['theta'] = np.where(
    df['Unemployment'] > 0,
    df['vacancies'] / df['Unemployment'],
    np.nan
)

# Quick check
df[['month', 'vacancies', 'Unemployment', 'theta']].head()


,month,vacancies,Unemployment,theta
0,2015-04,440425.0,1337.2,329.363596
1,2015-05,407040.0,1324.7,307.269570
2,2015-06,398910.0,1326.6,300.701040
3,2015-07,376570.0,1336.0,281.863772
4,2015-08,369190.0,1358.3,271.802989


In [9]:
# ============================================================
# US4 – TASK 4.3: OUTLIER INSPECTION FOR θ
# ============================================================

# Summary statistics
print("Theta summary statistics:")
print(df['theta'].describe())

# Identify extreme values using IQR (for inspection only)
Q1 = df['theta'].quantile(0.25)
Q3 = df['theta'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['theta'] < lower_bound) | (df['theta'] > upper_bound)
]

print("\nNumber of potential outlier months:", outliers.shape[0])
outliers[['month', 'theta']]


Theta summary statistics:
count    111.000000
mean     474.080945
std      186.912264
min      241.133824
25%      329.083471
50%      437.226340
75%      537.655887
max      976.705392
Name: theta, dtype: float64

Number of potential outlier months: 7


,month,theta
83,2022-03,882.940174
84,2022-04,877.339901
85,2022-05,910.998893
86,2022-06,962.069301
87,2022-07,976.705392
88,2022-08,872.331396
89,2022-09,887.197637


Outlier inspection was conducted on the θ indicator using interquartile range (IQR)
bounds. Extreme values were observed primarily during periods of sharp changes in
unemployment and job vacancies, particularly around the COVID-19 shock and post-
pandemic recovery.

These values were retained, as they reflect meaningful economic conditions rather
than data errors. No outlier removal or smoothing was applied at this stage.


In [10]:
# ============================================================
# US4 – TASK 4.4: SAVE FINAL MODELING DATASET (v3)
# ============================================================

df.to_csv(
    "../1_data/processed/final_dataset.csv",
    index=False
)

